In [ ]:
# 🧬⛳ NeuroGolf Offline Carry — 7225.89 LB

Thanks to the NeuroGolf community and all public notebook authors for the shared work that made this compact carry possible.

This version is deliberately offline and submission-focused: no online access, no external datasets, no extra downloads, and no visualization dependencies. It writes the proven 400-model ONNX archive as `submission.zip`, checks the archive hash, and verifies the canonical `task001.onnx` … `task400.onnx` layout before finishing.


In [ ]:
# English greeting + minimal offline environment check
import sys, platform

print("Thank you to the NeuroGolf community and all public notebook authors.")
print("Offline carry mode: no online access, no extra downloads, no external datasets.")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


In [ ]:
# Emit the validated NeuroGolf submission.zip offline.
# Uses only Python standard-library modules and embedded bytes.

import base64
import hashlib
import zipfile
from pathlib import Path

EXPECTED_SHA256 = "0f8832f2eb9c4d0367e22520d45e6d99b66ebc6d1b53a58e4db4e169ab0b87c9"
EXPECTED_BYTES = 469191
EXPECTED_NAMES = [f"task{i:03d}.onnx" for i in range(1, 401)]

SUBMISSION_ZIP_B64 = """
...
"""

payload = base64.b64decode("".join(SUBMISSION_ZIP_B64.split()))
actual_sha = hashlib.sha256(payload).hexdigest()

if actual_sha != EXPECTED_SHA256:
    raise RuntimeError(f"submission.zip SHA256 mismatch: {actual_sha}")
if len(payload) != EXPECTED_BYTES:
    raise RuntimeError(f"submission.zip byte-size mismatch: {len(payload)}")

work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
work.mkdir(parents=True, exist_ok=True)
submission_path = work / "submission.zip"
submission_path.write_bytes(payload)

with zipfile.ZipFile(submission_path, "r") as zf:
    names = sorted(zf.namelist())
    if names != EXPECTED_NAMES:
        raise RuntimeError("submission.zip must contain exactly task001.onnx ... task400.onnx")
    bad = [n for n in names if zf.getinfo(n).file_size <= 0]
    if bad:
        raise RuntimeError(f"empty ONNX files detected: {bad[:5]}")

print("submission.zip written:", submission_path)
print("models:", len(EXPECTED_NAMES))
print("bytes:", len(payload))
print("sha256:", actual_sha)
print("status: ready to submit")


In [ ]:
# Offline score-board: largest remaining cost targets.
# This does not change the archive; it gives the next micro-golf attack list.

import math

COSTS = {1: 445, 2: 14770, 3: 261, 4: 5137, 5: 6801, 6: 153, 7: 127, 8: 5626, 9: 6699, 10: 1012, 11: 1460, 12: 2259, 13: 1930, 14: 4119, 15: 900, 16: 10, 17: 7004, 18: 25317, 19: 2793, 20: 1526, 21: 324, 22: 2302, 23: 8996, 24: 110, 25: 10236, 26: 200, 27: 1310, 28: 168, 29: 5299, 30: 2396, 31: 688, 32: 910, 33: 1639, 34: 2365, 35: 2249, 36: 2177, 37: 4746, 38: 213, 39: 455, 40: 190, 41: 1789, 42: 3110, 43: 793, 44: 5617, 45: 1673, 46: 2285, 47: 862, 48: 804, 49: 345, 50: 3852, 51: 1744, 52: 194, 53: 30, 54: 25517, 55: 3144, 56: 34, 57: 668, 58: 1033, 59: 549, 60: 1010, 61: 1668, 62: 2536, 63: 1873, 64: 12687, 65: 641, 66: 12366, 67: 47, 68: 623, 69: 3912, 70: 3008, 71: 2698, 72: 421, 73: 40, 74: 9050, 75: 1487, 76: 15042, 77: 14830, 78: 1293, 79: 3133, 80: 10503, 81: 464, 82: 190, 83: 210, 84: 2038, 85: 5381, 86: 3136, 87: 5, 88: 1872, 89: 6772, 90: 3058, 91: 3028, 92: 6842, 93: 3559, 94: 2709, 95: 345, 96: 8356, 97: 910, 98: 900, 99: 1899, 100: 361, 101: 13773, 102: 2527, 103: 60, 104: 238, 105: 2943, 106: 626, 107: 3694, 108: 300, 109: 1650, 110: 10706, 111: 311, 112: 1774, 113: 30, 114: 900, 115: 1043, 116: 30, 117: 4089, 118: 12283, 119: 1363, 120: 910, 121: 328, 122: 810, 123: 1343, 124: 1953, 125: 3463, 126: 605, 127: 396, 128: 136, 129: 80, 130: 30, 131: 3978, 132: 4089, 133: 23260, 134: 2284, 135: 200, 136: 1194, 137: 3526, 138: 13011, 139: 904, 140: 5, 141: 1383, 142: 90, 143: 837, 144: 104, 145: 12855, 146: 267, 147: 532, 148: 6007, 149: 136, 150: 158, 151: 900, 152: 90, 153: 795, 154: 2759, 155: 142, 156: 1564, 157: 7032, 158: 30757, 159: 1568, 160: 1334, 161: 2306, 162: 4068, 163: 2345, 164: 30, 165: 6054, 166: 24, 167: 146, 168: 2239, 169: 1670, 170: 2545, 171: 910, 172: 30, 173: 13595, 174: 4559, 175: 1255, 176: 210, 177: 3621, 178: 763, 179: 0, 180: 210, 181: 368, 182: 6693, 183: 1025, 184: 2160, 185: 1961, 186: 103, 187: 16112, 188: 764, 189: 913, 190: 2527, 191: 11320, 192: 7237, 193: 910, 194: 949, 195: 921, 196: 4538, 197: 2602, 198: 12298, 199: 1829, 200: 756, 201: 3676, 202: 8644, 203: 355, 204: 10373, 205: 11209, 206: 4172, 207: 236, 208: 6083, 209: 7927, 210: 30, 211: 270, 212: 2360, 213: 1851, 214: 525, 215: 749, 216: 9127, 217: 1470, 218: 649, 219: 9647, 220: 900, 221: 805, 222: 6194, 223: 5, 224: 2831, 225: 1033, 226: 1633, 227: 100, 228: 962, 229: 200, 230: 900, 231: 219, 232: 1410, 233: 34481, 234: 6408, 235: 170, 236: 237, 237: 1838, 238: 1947, 239: 947, 240: 1475, 241: 0, 242: 432, 243: 22650, 244: 1179, 245: 2743, 246: 2637, 247: 469, 248: 151, 249: 261, 250: 2538, 251: 4920, 252: 180, 253: 500, 254: 687, 255: 10379, 256: 2251, 257: 400, 258: 160, 259: 894, 260: 1576, 261: 200, 262: 106, 263: 671, 264: 5408, 265: 4357, 266: 350, 267: 766, 268: 3464, 269: 709, 270: 3069, 271: 1187, 272: 402, 273: 2160, 274: 175, 275: 1356, 276: 10, 277: 3745, 278: 6130, 279: 5555, 280: 6035, 281: 2386, 282: 446, 283: 910, 284: 3649, 285: 19967, 286: 27338, 287: 1994, 288: 958, 289: 742, 290: 355, 291: 68, 292: 80, 293: 1270, 294: 910, 295: 1604, 296: 314, 297: 1389, 298: 135, 299: 50, 300: 528, 301: 1141, 302: 1774, 303: 1450, 304: 1441, 305: 636, 306: 300, 307: 5, 308: 1468, 309: 10, 310: 3287, 311: 30, 312: 20, 313: 188, 314: 100, 315: 230, 316: 426, 317: 146, 318: 100, 319: 5861, 320: 770, 321: 300, 322: 200, 323: 2177, 324: 8894, 325: 1483, 326: 30, 327: 640, 328: 6766, 329: 1059, 330: 3022, 331: 910, 332: 561, 333: 3223, 334: 197, 335: 3135, 336: 1820, 337: 10, 338: 9250, 339: 140, 340: 4714, 341: 1429, 342: 1084, 343: 1546, 344: 910, 345: 1495, 346: 1254, 347: 143, 348: 2095, 349: 15042, 350: 9036, 351: 1230, 352: 460, 353: 244, 354: 2692, 355: 2704, 356: 1319, 357: 291, 358: 4027, 359: 4344, 360: 340, 361: 4809, 362: 521, 363: 4617, 364: 20805, 365: 3932, 366: 34274, 367: 22037, 368: 6189, 369: 1393, 370: 9110, 371: 467, 372: 710, 373: 60, 374: 2284, 375: 327, 376: 186, 377: 8207, 378: 4437, 379: 9113, 380: 99, 381: 2064, 382: 5774, 383: 6038, 384: 509, 385: 30, 386: 211, 387: 5059, 388: 2191, 389: 130, 390: 3090, 391: 159, 392: 1951, 393: 144, 394: 2780, 395: 165, 396: 4934, 397: 2782, 398: 2855, 399: 60, 400: 1223}
top = sorted(COSTS.items(), key=lambda kv: kv[1], reverse=True)[:15]
total = sum(COSTS.values())

print("Local total cost ledger:", total)
print("Top remaining attack candidates:")
for rank, (task_id, cost) in enumerate(top, 1):
    points = max(1.0, 25.0 - math.log(cost))
    print(f"{rank:02d}  task{task_id:03d}  cost={cost:>6}  current_points≈{points:.4f}")

print("Next practical target: reduce one high-cost ONNX, then replace the embedded archive.")
